<a href="https://colab.research.google.com/github/mamathabaig/AI-lab/blob/main/1BM25CS476_Week8_FOL_Forward%20chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import copy
import itertools

# Helper functions to distinguish variables (lowercase) from constants/predicates (Uppercase)
def is_variable(term):
    return isinstance(term, str) and term[0].islower()

def parse_atom(atom_str):
    """Parses 'Predicate(arg1, arg2)' into ('Predicate', ['arg1', 'arg2'])"""
    pred, args = atom_str.split("(")
    args = [arg.strip() for arg in args[:-1].split(",")]
    return pred.strip(), args

def format_atom(pred, args):
    return f"{pred}({', '.join(args)})"

# --- UNIFICATION ENGINE ---
def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    if theta is None:
        return None
    elif x == y:
        return theta
    elif is_variable(x):
        return unify_var(x, y, theta)
    elif is_variable(y):
        return unify_var(y, x, theta)
    elif isinstance(x, list) and isinstance(y, list) and len(x) == len(y):
        return unify(x[1:], y[1:], unify(x[0], y[0], theta))
    else:
        return None

def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    elif x in theta:
        return unify(var, theta[x], theta)
    else:
        new_theta = theta.copy()
        new_theta[var] = x
        return new_theta

def subst(theta, atom_str):
    """Applies a substitution dictionary theta to an atomic sentence."""
    pred, args = parse_atom(atom_str)
    new_args = []
    for arg in args:
        while arg in theta:
            arg = theta[arg]
        new_args.append(arg)
    return format_atom(pred, new_args)

# --- VARIABLE STANDARDIZATION ---
counter = itertools.count(1)
def standardize_variables(premises, conclusion):
    """Renames variables in a rule so they don't clash across iterations."""
    mapping = {}
    idx = next(counter)

    def rename(atom_str):
        pred, args = parse_atom(atom_str)
        new_args = []
        for arg in args:
            if is_variable(arg):
                if arg not in mapping:
                    mapping[arg] = f"{arg}_{idx}"
                new_args.append(mapping[arg])
            else:
                new_args.append(arg)
        return format_atom(pred, new_args)

    return [rename(p) for p in premises], rename(conclusion)

# --- FORWARD CHAINING ALGORITHM (FOL-FC-ASK) ---
def match_premises(premises, kb, theta=None):
    """Recursively finds all substitutions theta that satisfy a list of premises."""
    if theta is None:
        theta = {}
    if not premises:
        yield theta
        return

    first, rest = premises[0], premises[1:]
    first_sub = subst(theta, first)
    pred1, args1 = parse_atom(first_sub)

    for fact in kb:
        pred2, args2 = parse_atom(fact)
        if pred1 == pred2:
            new_theta = unify(args1, args2, theta)
            if new_theta is not None:
                yield from match_premises(rest, kb, new_theta)

def fol_fc_ask(facts, rules, query):
    kb = set(facts)
    iteration = 0

    while True:
        iteration += 1
        new_facts = set()
        print(f"--- Iteration {iteration} ---")

        for premises, conclusion in rules:
            std_premises, std_conclusion = standardize_variables(premises, conclusion)

            for theta in match_premises(std_premises, kb):
                inferred = subst(theta, std_conclusion)

                if inferred not in kb and inferred not in new_facts:
                    new_facts.add(inferred)
                    print(f"Inferred: {inferred}")

                    # Check if the inferred fact unifies with the query
                    q_pred, q_args = parse_atom(query)
                    i_pred, i_args = parse_atom(inferred)
                    if q_pred == i_pred:
                        ans = unify(i_args, q_args)
                        if ans is not None:
                            return True, ans

        if not new_facts:
            break
        kb.update(new_facts)

    return False, None

# --- KNOWLEDGE BASE SETUP ---
if __name__ == "__main__":
    # Initial Atomic Facts (Constants start with Uppercase)
    initial_facts = [
        "American(Robert)",
        "Enemy(A, America)",
        "Owns(A, T1)",
        "Missile(T1)"
    ]

    # Definite Clauses: ([Premises], Conclusion) (Variables start with lowercase)
    rules = [
        (["American(p)", "Weapon(q)", "Sells(p, q, r)", "Hostile(r)"], "Criminal(p)"),
        (["Missile(x)", "Owns(A, x)"], "Sells(Robert, x, A)"),
        (["Missile(x)"], "Weapon(x)"),
        (["Enemy(x, America)"], "Hostile(x)")
    ]

    query = "Criminal(Robert)"
    print(f"Query: {query}\n")

    proved, substitution = fol_fc_ask(initial_facts, rules, query)
    print(f"\nResult: Query '{query}' proved? -> {proved}")

Query: Criminal(Robert)

--- Iteration 1 ---
Inferred: Sells(Robert, T1, A)
Inferred: Weapon(T1)
Inferred: Hostile(A)
--- Iteration 2 ---
Inferred: Criminal(Robert)

Result: Query 'Criminal(Robert)' proved? -> True
